# 03 — Model Building

This notebook defines the `SentimentLSTM` architecture, explains every design decision, and smoke-tests the model before any training occurs.

**Prerequisites:** `02_preprocessing.ipynb` must have been run first so that `data/vocab.pkl` exists.

**Notebook outline:**
1. Load Vocabulary
2. Define Model
3. Model Summary
4. Smoke Test
5. Experiment Variants (Baseline and 2-Layer Preview)

In [1]:
import sys, os

# ── Colab vs. local environment detection ───────────────────────────────
if os.path.exists("/content"):
    repo_root = "/content/IMDb_Sentiment_Analysis"
    if not os.path.exists(repo_root):
        os.system("git clone https://github.com/azrakarakaya1/IMDb_Sentiment_Analysis.git " + repo_root)
    os.chdir(repo_root)
else:
    os.chdir(os.path.abspath(".."))

sys.path.insert(0, os.path.abspath("."))
print(f"Working directory: {os.getcwd()}")
import torch
import pickle

Working directory: /Users/azrakarakaya/IMDb_Sentiment_Analysis


## Section 1 — Load Vocabulary

Before we can instantiate the model we need to know the vocabulary size. The `SentimentLSTM` embedding layer is a lookup table with one row per vocabulary token — so `vocab_size` directly determines the number of rows (and therefore the number of parameters) in that table.

We load the `Vocabulary` object that was serialised by `02_preprocessing.ipynb`. If the file is missing, run that notebook first.

In [2]:
from src.preprocessing import Vocabulary

# Load the vocabulary built from the training split in 02_preprocessing.ipynb.
# Vocabulary.load() deserialises the pickled Vocabulary object from disk.
vocab = Vocabulary.load('data/vocab.pkl')

# len(vocab) includes the two special tokens <PAD> (idx 0) and <UNK> (idx 1)
# plus up to 20,000 regular tokens, so the total is at most 20,002.
print(f'Vocabulary size: {len(vocab):,} tokens')

Vocabulary size: 20,002 tokens


**Why we need the vocab size to instantiate the model:**

The first layer of `SentimentLSTM` is an `nn.Embedding` table of shape `(vocab_size, embedding_dim)`. Each row stores the learned dense vector for one token. PyTorch allocates this table at construction time, so `vocab_size` must be known before training begins.

Using a vocab size that is too small would cause an index-out-of-range error at runtime whenever the model encounters a token index ≥ `vocab_size`. Using the exact size from the saved vocabulary guarantees that every index produced by the tokenizer is valid.

## Section 2 — Define Model

We import `SentimentLSTM` from `src.model` and instantiate the final architecture used in training:

| Hyperparameter | Value | Rationale |
|---|---|---|
| `vocab_size` | `len(vocab)` | Must match the tokenizer's vocabulary |
| `embedding_dim` | 128 | Standard starting point; balances capacity vs. training speed |
| `hidden_size` | 256 | Larger capacity needed for 3-layer architecture |
| `num_layers` | 3 | Final architecture; outperformed 1-layer (~87%) and 2-layer (~89.56%) |
| `dropout` | 0.4 | Applied between LSTM layers and before the classifier |
| `use_batch_norm` | `True` | Normalises the final hidden state; stabilises training |

In [3]:
from src.model import SentimentLSTM

# Instantiate the final trained model.
# These hyperparameters match the checkpoint saved by 04_training_and_evaluation.ipynb.
model = SentimentLSTM(
    vocab_size=len(vocab),    # embedding table has one row per token
    embedding_dim=128,        # each token maps to a 128-dimensional vector
    hidden_size=256,          # LSTM maintains a 256-dimensional hidden state
    num_layers=3,             # three stacked LSTM layers
    dropout=0.4,              # 40% dropout between layers and before classifier
    use_batch_norm=True,      # normalise final hidden state before Linear
)

print(model)

**Purpose of each architectural component:**

- **Embedding layer** (`nn.Embedding`): maps integer token indices to dense 128-dimensional vectors. Each token gets its own learnable vector that captures semantic meaning. `padding_idx=0` means the `<PAD>` token always produces a zero vector and receives no gradient updates — padding tokens should not influence the learned representations.

- **LSTM layers** (`nn.LSTM`, 3 stacked): each layer processes the output of the previous layer left-to-right, building progressively more abstract representations. Layer 1 captures local word patterns, layer 2 phrase-level structure, and layer 3 discourse-level sentiment. We take the final hidden state of the *top* layer as a fixed-size summary of the entire review. Inter-layer dropout (0.4) is applied between layers by PyTorch automatically when `num_layers > 1`.

- **Batch Normalisation** (`nn.BatchNorm1d`): normalises the final hidden state across the batch before the classifier. This reduces internal covariate shift, stabilises training in the early epochs, and acts as a mild regulariser that complements dropout.

- **Dropout layer** (`nn.Dropout`): randomly zeros 40% of the hidden-state activations during training. Disabled automatically during evaluation (`model.eval()`).

- **Linear (logit output)** (`nn.Linear`): projects the 256-dimensional hidden state to a single scalar logit. No activation is applied here — the raw logit is the model's output. During training, `BCEWithLogitsLoss` applies the sigmoid internally for numerical stability. For inference, `torch.sigmoid(logit) >= 0.5` → positive.

## Section 3 — Model Summary

We print a summary of each named module along with its parameter count, then compute the total and trainable parameter counts. This gives us a clear picture of where the model's capacity is concentrated.

In [4]:
# Print a dependency-free model summary: layer name and parameter count.
print(f"{'Layer':<30} {'Parameters':>15}")
print("-" * 47)

for name, module in model.named_modules():
    # named_modules() recurses into sub-modules; skip the top-level container
    if name == '':
        continue
    # Count parameters belonging directly to this module (not its children)
    params = sum(p.numel() for p in module.parameters(recurse=False))
    if params > 0:
        print(f"{name:<30} {params:>15,}")

print("-" * 47)

# Total and trainable parameter counts across the entire model
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"{'Total parameters':<30} {total_params:>15,}")
print(f"{'Trainable parameters':<30} {trainable_params:>15,}")

Layer                               Parameters
-----------------------------------------------
embedding                            2,560,256
lstm                                    49,664
fc                                          65
-----------------------------------------------
Total parameters                     2,609,985
Trainable parameters                 2,609,985


**Interpreting the parameter count:**

The parameter budget breaks down as follows for the final model (vocab_size = 20,002, hidden_size = 256):

| Layer | Formula | Count |
|---|---|---|
| Embedding | `vocab_size × embedding_dim` | 20,002 × 128 = **2,560,256** |
| LSTM Layer 1 | `4 × (emb_dim + hidden) × hidden + 8 × hidden` | 4×(128+256)×256 + 2,048 = **395,264** |
| LSTM Layers 2–3 | `4 × (hidden + hidden) × hidden + 8 × hidden` each | 2 × (4×512×256 + 2,048) = **1,052,672** |
| BatchNorm | `2 × hidden_size` (weight + bias) | **512** |
| Linear | `hidden_size + 1` (weight + bias) | **257** |

The embedding layer still dominates (≈ 64% of all parameters), but the three-layer LSTM now contributes a meaningful share (≈ 36%). The total parameter count is roughly 4× larger than the single-layer baseline, which is why dropout, batch normalisation, and weight decay are all necessary to prevent overfitting.

## Section 4 — Smoke Test

Before training, we verify that the model accepts the correct input shape and produces valid probability outputs. We create a dummy batch of 4 reviews, each represented as a sequence of 500 random token indices, and run a forward pass.

In [5]:
# Set the model to evaluation mode so dropout is disabled during the smoke test.
# (We want deterministic output for verification, not stochastic dropout.)
model.eval()

# Create a dummy input: batch of 4 reviews, each with 500 token indices.
x = torch.randint(0, len(vocab), (4, 500))

# Run the forward pass inside torch.no_grad() to skip gradient computation.
with torch.no_grad():
    output = model(x)

# --- Assertions ---
# The output must have shape (batch_size, 1): one logit per review.
assert output.shape == (4, 1), f"Expected shape (4, 1), got {output.shape}"

# The model outputs raw logits (unbounded reals), not probabilities.
# We verify that all values are finite (no NaN / Inf from bad wiring).
assert output.isfinite().all(), "Output contains NaN or Inf — check model wiring"

print("Smoke test passed!")
print(f"Output shape   : {output.shape}")
print(f"Output (logits): {output.squeeze().tolist()}")
print(f"Output (probs) : {torch.sigmoid(output).squeeze().tolist()}")

**What the smoke test verifies:**

The smoke test checks three things before any training has occurred:

1. **Correct input shape handling**: the model accepts a `(batch_size, seq_len)` integer tensor without errors. This confirms that the embedding lookup, stacked LSTM layers, batch normalisation, and linear layer are wired together correctly.

2. **Correct output shape**: the output is `(batch_size, 1)` — one scalar per review. This is the shape expected by `BCEWithLogitsLoss` during training.

3. **Finite output values**: the model outputs raw logits (unbounded real numbers, not probabilities). We check that all values are finite — no `NaN` or `Inf`. With random weights the logits will typically be small numbers close to 0; after applying `torch.sigmoid()` they fall near 0.5, consistent with a randomly initialised model that has no preference for either class.

Running this test before training catches wiring bugs early, when they are easiest to fix.

## Section 5 — Experiment Variants (Baseline and 2-Layer Preview)

In `04_training_and_evaluation.ipynb` we train three configurations and compare their validation accuracy:

| Configuration | hidden_size | num_layers | dropout | BatchNorm | Val Acc |
|---|---|---|---|---|---|
| Baseline | 64 | 1 | 0.5 | No | ~87% |
| Experiment | 128 | 2 | 0.5 | No | ~89.56% |
| **Final** | **256** | **3** | **0.4** | **Yes** | **90.52%** |

Here we instantiate the baseline and the 2-layer experiment to compare their parameter counts against the final model defined in Section 2.

**Hypothesis going into training:** each additional layer should improve the model's ability to capture long-range dependencies. The BatchNorm addition should stabilise training for the deeper architecture.

In [6]:
# ── Baseline (1-layer, hidden=64, no BatchNorm) ──────────────────────────
model_baseline = SentimentLSTM(
    vocab_size=len(vocab), embedding_dim=128,
    hidden_size=64, num_layers=1, dropout=0.5, use_batch_norm=False,
)

# ── 2-Layer Experiment (2-layer, hidden=128, no BatchNorm) ───────────────
model_2layer = SentimentLSTM(
    vocab_size=len(vocab), embedding_dim=128,
    hidden_size=128, num_layers=2, dropout=0.5, use_batch_norm=False,
)

# ── Smoke-test all three variants ────────────────────────────────────────
for tag, m in [("Baseline", model_baseline), ("2-layer experiment", model_2layer), ("Final (3-layer+BN)", model)]:
    m.eval()
    with torch.no_grad():
        out = m(x)
    assert out.shape == (4, 1)
    assert out.isfinite().all()
    print(f"{tag:<25} shape={out.shape}  logit range=[{out.min().item():.3f}, {out.max().item():.3f}]")

# ── Parameter count comparison ───────────────────────────────────────────
print()
configs = [
    ("Baseline (1L h=64)",     model_baseline),
    ("Experiment (2L h=128)",  model_2layer),
    ("Final (3L h=256 +BN)",   model),
]
for tag, m in configs:
    p = sum(par.numel() for par in m.parameters())
    print(f"{tag:<28}: {p:>10,} parameters")

**Why the architecture evolved from 1 → 2 → 3 layers:**

1. **`num_layers=1 → 2`**: a second LSTM layer receives the full output sequence from layer 1 and learns higher-level representations. This is analogous to `return_sequences=True` in Keras. PyTorch's `nn.LSTM` handles the inter-layer connection automatically. Hypothesis confirmed: the 2-layer model achieved ~89.56% validation accuracy vs. ~87% for the baseline.

2. **`num_layers=2 → 3` + Batch Normalisation**: a third layer was added to capture discourse-level sentiment patterns (e.g., overall tone, narrative arc). `hidden_size` was increased from 128 to 256 to give each layer enough capacity across 500 tokens. Batch normalisation was added on the final hidden state to stabilise the deeper architecture — especially useful in early epochs when the running statistics are still settling. The 3-layer model achieved **90.52% validation accuracy** — a 1% gain over the 2-layer experiment.

**Parameter growth:** the embedding table (unchanged across all three) still accounts for the majority of parameters. The LSTM layers grow significantly with depth and hidden size, which is why regularisation was strengthened at each step (dropout 0.4, BatchNorm, weight decay 1e-4 in the Adam optimiser).

Full training curves and a per-epoch comparison are in `04_training_and_evaluation.ipynb`.